In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
## general libraries
import pathlib
from rich.pretty import install, pprint

## data handling libraries
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.dask import TqdmCallback as ProgressBarDask

# plotting libraries
import matplotlib.pyplot as plt

## machine learning libraries
import gpytorch 
import torch

## PAMIR libraries
import plots
import mlflow
import pamir_mlpermafrost as pamir
from cryogrid_pytools import xr_raster_vector

install(overflow=True)

# Load data

In [ ]:
store_ground_temp = "simplecache::s3://spi-pamir-cryogrid/pamir-MLpermafrost/data-output/ground_tempSEAS_DEPTH-x7129e34a-covar_poly1_rbf_index-likelihood_gaussian_gt02lt5-opt_adam_lr0.03-seed42.zarr/"
ds_ground_temp = xr.open_zarr(store_ground_temp, storage_options=pamir.data.s3_utils.get_fsspec_kwargs())

store_spatial = "simplecache::s3://spi-pamir-cryogrid/pamir-MLpermafrost/data-inference/inference_variables-710w365s750e400n-100m.zarr/"
ds_spatial = xr.open_zarr(store_spatial, storage_options=pamir.data.s3_utils.get_fsspec_kwargs())

da_mask = ds_spatial.surface_index > 0
lakes = ds_spatial.land_cover == 1
snow = (ds_spatial.land_cover == 9).compute().morph.clean()
invalid = ds_spatial.land_cover.round().isin([1, 2, 3, 4, 5, 6, 7]).compute().morph.clean()

from pamir_mlpermafrost.utils.chained_upath import ChainedUPath as UPath
path = UPath(store_ground_temp, storage_options=pamir.data.s3_utils.get_fsspec_kwargs())

In [ ]:
da = (
    ds_ground_temp.yhat_avg_ground_temp
    .sel(season='JAS')
    .where(~invalid & ~lakes & ~snow)
    .coarsen(x=5, y=5, boundary='pad').mean()
    .compute()
)

In [ ]:
seas = da.season.item()

fig, axs, cbar = plots.plot_depths(
    da, 
    path_to_zarr=store_ground_temp, 
    title=f'Ground Temperature for $\\mathbf{{\\overline{{{seas}}}}}$ [2000-2024]',
    info='Point locations modelled with CryoGrid clustering approach interpolated to 100 m resolution with Gaussian Processes',
    cmap='RdBu_r', vmin=-10, vmax=10)

cbar.set_label('Ground Temperature [°C]')

fig.savefig(f'./figs/gp_ground_temp_{seas}.png', dpi=300, bbox_inches='tight', transparent=True, facecolor='none')

# Permafrost depth (i.e., active layer thickness)

In [ ]:
summer = ds_ground_temp.yhat_avg_ground_temp.sel(season='JAS')
winter = ds_ground_temp.yhat_avg_ground_temp.sel(season='JFM')

summer_interp = summer.interp(depth=np.arange(0.25, 7.25, 0.125), method='linear')

summer_frozen = summer < 0
summer_interp_frozen = summer_interp < 0
winter_frozen = winter < 0

In [ ]:
mask = ~invalid & ~lakes & ~snow
da = ds_ground_temp.yhat_std_ground_temp.sel(season='JAS').where(mask).compute()

In [ ]:
permafrost = (summer_frozen & winter_frozen).any(dim='depth').pipe(lambda x: x & ~invalid).compute().morph.clean()

In [ ]:
with ProgressBarDask():
    da = (
        summer_interp_frozen
        .idxmax(dim='depth')
        .persist()
        .where(~snow)
        .fillna(0.20)
        .where(permafrost)
        .drop_vars('season')
        .rolling(x=3, y=3, center=True, min_periods=1).median()
        .rolling(x=3, y=3, center=True, min_periods=1).median()
        .compute()
    )

In [ ]:
cmap = plt.cm.GnBu_r
cmap.set_under("#7c2f5f")
cmap.set_bad('none')

fig, axs = plt.subplots(figsize=[8, 6], dpi=300)
img = da.plot.imshow(
    vmin=0.24, 
    vmax=4, 
    cmap=cmap, 
    ax=axs, 
    cbar_kwargs=dict(extendrect=True, extendfrac=0.08, pad=0.02))

cbar = img.colorbar
cbar.ax.axhline(0.24, color='w', lw=4, zorder=10)
cbar.set_ticks([0.24, 1, 2, 3, 4])
cbar.set_ticklabels(['\nPermanent\nSnow', '1', '2', '3', '4'])
cbar.set_label('Depth of permafrost [m]', rotation=90, labelpad=-35, size=10)

axs.set_title('Depth of permafrost based on summer ground temperatures', loc='left')
axs.set_xlabel('')

fig.subplots_adjust(hspace=0.08)

plots.add_plot_meta(
    axs, 
    info='CryoGrid point locations mapped at 100 m using Gaussian Processes.\n         Reported depth is the freeze-thaw boundary depth.',
    path=store_ground_temp,
    size=6,
)

fig.savefig('./figs/gp_depth_of_permafrost.png', bbox_inches='tight', dpi=300, facecolor='none', transparent=True)

# Summer-winter difference

In [ ]:
da = (
    ds_ground_temp.yhat_avg_ground_temp
    .where(da_mask)
    .sel(season=['JFM', 'JAS'])
    .diff('season')
    .squeeze(drop=True)
    .rename('ground_temp_JAS_minus_JFM')
    .coarsen(x=4, y=4, boundary='pad').mean()
    .persist())

In [ ]:
fig, axs, cbar = plots.plot_depths(
    da, 
    store_ground_temp, 
    title='Ground Temperature Difference (JAS - JFM)', 
    info='Point locations modelled with CryoGrid clustering approach interpolated to 100 m resolution with Gaussian Processes', 
    vmin=-2, vmax=17, cmap='Spectral_r')

cbar.set_label('JAS $-$ JFM Ground Temperature difference [°C]')
fig.savefig('./figs/gp_ground_temp_JAS_minus_JFM.png', dpi=300, bbox_inches='tight', transparent=True, facecolor='none')